In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/train.csv')
oil = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/oil.csv')
stores = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/stores.csv')
holidays = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv')
transactions = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/transactions.csv')
import matplotlib.pyplot as plt
import seaborn as sns
#train.head()
#oil.head()
#stores.head()
#holidays.head()

In [ ]:
#stores.head()

In [ ]:
#train['sales'].value_counts()

欠損値の数について調べる

In [ ]:
#train.isnull().sum()

fimily属性について考える

In [ ]:
#sns.stripplot(x='family', y='sales', data=train)

In [ ]:
#train['family'].value_counts()

In [ ]:
#sns.stripplot(x='store_nbr', y='sales', data=train)

In [ ]:
#sns.stripplot(x='date', y='sales', data=train)

In [ ]:
#sns.stripplot(x='onpromotion', y='sales', data=train)

In [ ]:
#sns.stripplot(x='store_nbr', y='onpromotion', data=train)

In [ ]:
#train['onpromotion'].value_counts()

In [ ]:
#train.describe()

"store_nbr","date"と"family"をダミー変数化

データ整理

In [ ]:
holidays["is_holiday"] = holidays["type"].isin(["Holiday", "Additional", "Bridge"])
# イベントは個別の特徴
holidays["is_event"] = holidays["type"] == "Event"
#stores.drop(['id'],axis = 1, inplace = True)
merged_s = pd.merge(train, stores, on="store_nbr",how="left")
merged_so = pd.merge(merged_s, oil, on="date",how="left")
merged_soh = pd.merge(merged_so, holidays, on="date",how="left")
merged = pd.merge(merged_soh, transactions, on=["date","store_nbr"],how="left")
merged.head()

In [ ]:
# 欠損値を補う
# dcoilwticoの補完、前方補完(直前のデータで補完)、直前のデータがない場合は後方補完
merged["dcoilwtico"] = merged["dcoilwtico"].fillna(method="ffill").fillna(method="bfill")
# transactions(取引数)がNaN(欠損)のときは取引数0であるため変換
merged["transactions"] = merged["transactions"].fillna(0)
# is_holiday,is_eventが欠損の場合は普通の日のためFalseに変換
merged[["is_holiday", "is_event"]] = merged[["is_holiday", "is_event"]].fillna(False)
# transferredが欠損のとき,False
merged["transferred"] = merged["transferred"].fillna(False)
# 表示
merged.head()
# date分割
# dateをdatetime型に変換
merged["date"] = pd.to_datetime(merged["date"])

# 年、月、日をそれぞれの新しい列に分ける
merged["year"] = merged["date"].dt.year
merged["month"] = merged["date"].dt.month
merged["day"] = merged["date"].dt.day

# 曜日（0=月曜日）
merged["dayofweek"] = merged["date"].dt.dayofweek
# 休日(土日)かどうか
merged["is_weekend"] = merged["dayofweek"] >= 5
# 周番号(必要に応じて)
# merged["weekofyear"] = merged["date"].dt.isocalendar().week

# 表示
merged.head()
# city,stateとlocale,locale_nameを用いて、店舗の地域がholiday,eventの対象か確認、対象でない場合Falseに変更
merged.loc[(merged["locale"] == "Local") & (merged["city"] != merged["locale_name"]),["is_holiday","is_event"]] = False
merged.loc[(merged["locale"] == "Regional") & (merged["state"] != merged["locale_name"]),["is_holiday","is_event"]] = False
# 不必要なデータ削除
merged.drop(['id','locale','locale_name','city','state','type_y','description'], axis=1, inplace=True)
# 表示
merged.isnull().sum()
#merged.head()

In [ ]:
training["date"] = merged["date"]

X = training.drop("sales", axis=1)
y = training["sales"]

X["date"] = pd.to_datetime(training["date"])
X_train = X[X["date"] < "2017-07-01"].drop("date", axis=1)
y_train = y[X["date"] < "2017-07-01"]
X_valid = X[X["date"] >= "2017-07-01"].drop("date", axis=1)
y_valid = y[X["date"] >= "2017-07-01"]

ランダムフォレストの学習と予測を実施

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# 学習用と検証用の分割（2017-07-01で区切る）
X["date"] = pd.to_datetime(training["date"])
X_train = X[X["date"] < "2017-07-01"].drop("date", axis=1)
y_train = y[X["date"] < "2017-07-01"]
X_valid = X[X["date"] >= "2017-07-01"].drop("date", axis=1)
y_valid = y[X["date"] >= "2017-07-01"]

# モデル学習
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# バリデーション評価
y_pred = model.predict(X_valid)
mae = mean_absolute_error(y_valid, y_pred)
print(f"🌲 ランダムフォレスト MAE: {mae:.4f}")

テストデータを作成

In [ ]:
test = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/test.csv')

In [ ]:
merged_ts = pd.merge(test, stores, on="store_nbr",how="left")
merged_tso = pd.merge(merged_ts, oil, on="date",how="left")
merged_tsoh = pd.merge(merged_tso, holidays, on="date",how="left")
merged_t = pd.merge(merged_tsoh, transactions, on=["date","store_nbr"],how="left")

merged_t.head()

In [ ]:
# 欠損値を補う
# dcoilwticoの補完、前方補完(直前のデータで補完)、直前のデータがない場合は後方補完
merged_t["dcoilwtico"] = merged_t["dcoilwtico"].fillna(method="ffill").fillna(method="bfill")
# transactions(取引数)がNaN(欠損)のときは取引数0であるため変換
merged_t["transactions"] = merged_t["transactions"].fillna(0)
# is_holiday,is_eventが欠損の場合は普通の日のためFalseに変換
merged_t[["is_holiday", "is_event"]] = merged_t[["is_holiday", "is_event"]].fillna(False)
# transferredが欠損のとき,False
merged_t["transferred"] = merged_t["transferred"].fillna(False)
# 表示
merged_t.head()

In [ ]:
# date分割
# dateをdatetime型に変換
merged_t["date"] = pd.to_datetime(merged_t["date"])

# 年、月、日をそれぞれの新しい列に分ける
merged_t["year"] = merged_t["date"].dt.year
merged_t["month"] = merged_t["date"].dt.month
merged_t["day"] = merged_t["date"].dt.day

# 曜日（0=月曜日）
merged_t["dayofweek"] = merged_t["date"].dt.dayofweek
# 休日(土日)かどうか
merged_t["is_weekend"] = merged_t["dayofweek"] >= 5
# 周番号(必要に応じて)
# merged["weekofyear"] = merged["date"].dt.isocalendar().week

# 表示
merged_t.head()

In [ ]:
# city,stateとlocale,locale_nameを用いて、店舗の地域がholiday,eventの対象か確認、対象でない場合Falseに変更
merged_t.loc[(merged_t["locale"] == "Local") & (merged_t["city"] != merged_t["locale_name"]),["is_holiday","is_event"]] = False
merged_t.loc[(merged_t["locale"] == "Regional") & (merged_t["state"] != merged_t["locale_name"]),["is_holiday","is_event"]] = False
# 表示
merged_t.head()

In [ ]:
testing = pd.get_dummies(merged_t, columns=["store_nbr","family","type_x"], drop_first=True)

# 学習データと同じ列に揃える（不足カラムを補う）
missing_cols = set(X.columns) - set(testing.columns)
for col in missing_cols:
    testing[col] = 0
testing = testing[X.columns]  # 列順も合わせる

以上で前処理収量

storesのデータ（店舗に関する情報）を読み込む。

In [ ]:
stores = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/stores.csv')
import matplotlib.pyplot as plt
import seaborn as sns
stores.head()

storesのデータについて、店舗タイプ別の店舗数を調べる。

In [ ]:
print(stores["type"].value_counts())

trainsのデータ（エクアドルの複数店舗・複数商品における日次売上記録）を読み込む

In [ ]:
train = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/train.csv')
import matplotlib.pyplot as plt
import seaborn as sns
train.head()

店舗タイプごとの平均売り上げを比較し、店舗タイプが売り上げに影響するのかを調べる。

In [ ]:
merged = pd.merge(train, stores, on="store_nbr")
print(merged.groupby("type")["sales"].mean())

グラフに可視化する

In [ ]:
import matplotlib.pyplot as plt

type_sales = merged.groupby("type")["sales"].mean()

type_sales.plot(kind="bar",title="Average Sales by Store Type")
plt.xlabel("Store Type")
plt.ylabel("Average Sales")
plt.xticks(rotation=45)
plt.grid(True)
plt.show()

これより、店舗タイプの違いは、売り上げに影響すると予測した。これを確認する。まず、testのデータを読み込む

In [ ]:
test = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/test.csv')

次に、testにもstore情報（type）を付ける

In [ ]:
test = pd.merge(test, stores, on="store_nbr")

次に、店舗タイプに応じた売上予測を割り当てる

In [ ]:
test["sales"] = test["type"].map(type_sales)
test["sales"] = test["sales"].fillna(0.0)

次に、Cityごとの売り上げを比較してグラフに可視化する

In [ ]:
merged = pd.merge(train, stores, on="store_nbr")
city_sales = merged.groupby("city")["sales"].mean()

city_sales.sort_values(ascending=False).plot(kind="bar", title="Average Sales by City")
plt.xlabel("City")
plt.ylabel("Average Sales")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Cityごとの平均売上を計算する
city_sales = merged.groupby("city")["sales"].mean()

テストデータに対して予測

In [ ]:
test["sales"] = model.predict(testing)

最後に、提出ファイルを作成する。

In [ ]:
submission = test[["id", "sales"]]
submission.to_csv("submission.csv", index=False)